In [20]:
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
from dotenv import load_dotenv
import os
import glob

# Load API Key

In [21]:
load_dotenv()
api_key = os.getenv("KAKAO_API_KEY")

# Load raw csv data

In [22]:

# 사용할 컬럼만 지정 (NO 컬럼 제외)
columns_to_use = [
    '전용면적(㎡)', '계약년월', '계약일',
    '거래금액(만원)', '건축년도', '도로명'
]

# 병합할 CSV 파일이 들어있는 폴더 경로
folder_path = "../data/raw/apt_sale"

# 해당 경로의 모든 .csv 파일 리스트 얻기
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# 병합할 데이터프레임 저장할 리스트
df_list = []

# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")

# 데이터프레임 병합
df = pd.concat(df_list, ignore_index=True)

# 병합된 결과 출력
print(f"\n 병합 완료: 총 {len(df)}건")


 병합 완료: 총 512361건


# 면적당 단가 계산

In [23]:
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)
df['면적당 단가(만원)'] = df['거래금액(만원)'] / df['전용면적(㎡)']

# 아파트 나이 계산

In [30]:
df['계약년도'] = df['계약년월'].astype(str).str[:4].astype(int)
df['계약월'] = df['계약년월'].astype(str).str[5:6].astype(int)
df['아파트 나이'] = df['계약년도'] - df['건축년도']

# 도로명 기준 그룹 처리

In [25]:
# # === 도로명 기준 그룹 처리 === #
# def process_group(group):
#     if (group['아파트 나이'] <= 10).all():
#         row = group.iloc[0].copy()
#         row['면적당 단가(만원)'] = group['면적당 단가(만원)'].mean()
#         return pd.DataFrame([row])
#     else:
#         min_age = group['아파트 나이'].min()
#         return group[group['아파트 나이'] == min_age].iloc[[0]]

# df = df.groupby('도로명', group_keys=False).apply(process_group).reset_index(drop=True)


# 좌표 변환

In [26]:

# # === 좌표 변환 === #
# headers = {'Authorization': f'KakaoAK {api_key}'}

# def get_coords(address):
#     res = requests.get(
#         "https://dapi.kakao.com/v2/local/search/address.json",
#         headers=headers,
#         params={'query': address}
#     )
#     if res.status_code == 200 and res.json()['documents']:
#         doc = res.json()['documents'][0]
#         return doc['x'], doc['y']
#     return None, None

# longitudes, latitudes = [], []
# for address in tqdm(df['도로명'], desc="좌표 변환 중"):
#     x, y = get_coords(address)
#     longitudes.append(x)
#     latitudes.append(y)

# df['경도'] = longitudes
# df['위도'] = latitudes

# # === 최종 정제 및 저장 === #
# final_df = df[['위도', '경도', '건축년도', '구', '면적당 단가(만원)']].dropna()
# final_df['면적당 단가(만원)'] = np.log(final_df['면적당 단가(만원)'])

# # 디렉토리 없으면 생성
# OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# final_df.to_csv(OUTPUT_PATH, index=False)

# print(f"✅ 저장 완료: {OUTPUT_PATH}")

# 사용 안하는 컬럼 삭제하기

In [31]:
df.head()

,전용면적(㎡),계약년월,계약일,거래금액(만원),건축년도,도로명,면적당 단가(만원),계약년도,아파트 나이,계약월
0,35.300,202109,1,40000,1988,중랑천로 20,1133.144476,2021,33,9
1,43.350,202109,1,45000,1988,시루봉로 107,1038.062284,2021,33,9
2,114.912,202109,1,116500,2003,숭인로2길 61,1013.819270,2021,18,9
3,78.840,202109,1,64000,2003,긴고랑로14길 63,811.770675,2021,18,9
4,84.860,202109,1,168000,1996,광나루로56길 32,1979.731322,2021,25,9


In [35]:
df.drop(['계약년월','계약일','거래금액(만원)','건축년도','계약년도'],axis=1)

,전용면적(㎡),도로명,면적당 단가(만원),아파트 나이,계약월
0,35.300,중랑천로 20,1133.144476,33,9
1,43.350,시루봉로 107,1038.062284,33,9
2,114.912,숭인로2길 61,1013.819270,18,9
3,78.840,긴고랑로14길 63,811.770675,18,9
4,84.860,광나루로56길 32,1979.731322,25,9
...,...,...,...,...,...
512356,84.830,개포로 264,3182.836261,4,6
512357,49.860,개포로109길 9,2988.367429,32,6
512358,74.300,서운로 200,2954.239569,18,6
512359,59.970,서운로 62,3968.650992,4,6
